In [1]:
import sys
sys.path.append("/home/user/output_feedback/nhholyap/examples/")

import jax
import jax.numpy as jnp
import immrax as irx
from functools import partial
from jax import jit
from admire import AdmireNineDoFLinAct
from faulty_car.interval_functions import overlap_size_lax_scaled


In [2]:
admire = AdmireNineDoFLinAct()
x0_admire = jnp.zeros(9)
x0_admire = x0_admire.at[0].set(343 * 0.3)  # Set the first joint to 1.0
x0_interval_admire = irx.icentpert(x0_admire, jnp.ones(9) * 0.01)
admire_embsys = irx.natemb(admire)
p_rc_fault = jnp.array([0., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
p_rc_ivl = irx.icentpert(p_rc_fault, jnp.zeros_like(p_rc_fault))
p_lc_fault = jnp.array([1., 0., 1., 1., 1., 1., 1., 1., 1., 1.])
p_lc_ivl = irx.icentpert(p_lc_fault, jnp.zeros_like(p_lc_fault))
p_roe_fault = jnp.array([1., 1., 0., 1., 1., 1., 1., 1., 1., 1.])
p_roe_ivl = irx.icentpert(p_roe_fault, jnp.zeros_like(p_rc_fault))
p_rie_fault = jnp.array([1., 1., 1., 0., 1., 1., 1., 1., 1., 1.])
p_rie_ivl = irx.icentpert(p_rie_fault, jnp.zeros_like(p_rc_fault))
p_lie_fault = jnp.array([1., 1., 1., 1., 0., 1., 1., 1., 1., 1.])
p_lie_ivl = irx.icentpert(p_lie_fault, jnp.zeros_like(p_rc_fault))
p_loe_fault = jnp.array([1., 1., 1., 1., 1., 0., 1., 1., 1., 1.])
p_loe_ivl = irx.icentpert(p_loe_fault, jnp.zeros_like(p_rc_fault))
p_rudder_fault = jnp.array([1., 1., 1., 1., 1., 1., 0., 1., 1., 1.])
p_flap_fault = jnp.array([1., 1., 1., 1., 1., 1., 1., 0., 1., 1.])
p_flap_ivl = irx.icentpert(p_flap_fault, jnp.zeros_like(p_flap_fault))
p_yaw_tv_fault = jnp.array([1., 1., 1., 1., 1., 1., 1., 1., 0., 1.]) #Full loss of yaw thrust vectoring
p_pitch_tv_fault = jnp.array([1., 1., 1., 1., 1., 1., 1., 1., 1., 0.])
p_no_fault = jnp.array([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.]) #No fault
p_nominal_ivl = irx.icentpert(p_no_fault, jnp.zeros_like(p_no_fault))
p_yaw_tv_ivl = irx.icentpert(p_yaw_tv_fault, jnp.zeros_like(p_yaw_tv_fault))
p_rudder_ivl = irx.icentpert(p_rudder_fault, jnp.zeros_like(p_rudder_fault))
p_pitch_tv_ivl = irx.icentpert(p_pitch_tv_fault, jnp.zeros_like(p_pitch_tv_fault))
dt = 0.1
num_steps = 10


In [3]:
def propagate_interval_euler_admire(x_ivl, u_ol, p_ivl, dt=0.01):
    return irx.ut2i(admire_embsys.f(
        0, 
        irx.i2ut(x_ivl),
        u_ol,
        p_ivl,
    )) * dt + x_ivl


In [4]:
def _calculate_pairwise_loss(all_ivls, scaling_divisor):
    """Helper to calculate the sum of pairwise overlaps for a set of intervals."""
    num_scenarios = jax.tree_util.tree_leaves(all_ivls)[0].shape[0]
    i_indices, j_indices = jnp.triu_indices(num_scenarios, k=1)
    
    ivls_i = jax.tree_util.tree_map(lambda leaf: leaf[i_indices], all_ivls)
    ivls_j = jax.tree_util.tree_map(lambda leaf: leaf[j_indices], all_ivls)
    scaling_divisors = jnp.broadcast_to(
        scaling_divisor, 
        (i_indices.shape[0], *scaling_divisor.shape)
    )
    
    pairwise_losses = jax.vmap(overlap_size_lax_scaled, in_axes=(0, 0, 0))(ivls_i, ivls_j, scaling_divisors)
    return jnp.sum(pairwise_losses)

def loss_ff_staged_product_jax(u_ols, x_interval, p_nominal, p_faults, dt, num_u_steps):
    """
    Calculates a staged distinguishability loss.

    This function simulates a nominal and multiple fault scenarios. It applies a
    sequence of `N` open-loop controls (`u_ols`), each for `num_u_steps`.
    
    After each control stage, it calculates the total pairwise overlap between all
    scenarios (nominal + faults). The final loss is the *product* of these `N`
    intermediate loss values. A lower final loss is better, indicating that
    the states remained distinguishable at each stage.

    Args:
        u_ols (PyTree): A PyTree of JAX arrays, where each leaf has a leading
            dimension `N`, representing the sequence of N control inputs.
        x_interval (PyTree): The initial state interval.
        p_nominal (PyTree): Parameters for the nominal system.
        p_faults (PyTree): A PyTree of JAX arrays, where each leaf has a leading
            dimension `F`, representing F different fault scenarios.
        dt (float): The simulation time step.
        num_u_steps (int): The number of simulation steps per control stage.

    Returns:
        float: The final scalar loss, which is the product of stage-wise losses.
    """

    # Calculate Scaling factor
    scaling_divisors = x_interval.upper - x_interval.lower
    
    # Add a small epsilon to prevent division by zero if an initial interval has zero width.
    # scaling_divisors = jnp.maximum(scaling_divisors, 1e-6)

    num_faults = jax.tree_util.tree_leaves(p_faults)[0].shape[0]
    num_scenarios = 1 + num_faults

    # 1. Combine nominal and fault parameters into a single Pytree
    # This makes it easy to vmap the simulation over all scenarios.
    p_nominal_expanded = jax.tree_util.tree_map(lambda p: jnp.expand_dims(p, axis=0), p_nominal)
    all_params = jax.tree_util.tree_map(
        lambda n, f: jnp.concatenate([n, f], axis=0),
        p_nominal_expanded, p_faults
    )

    # 2. Prepare the initial state for all scenarios
    # All scenarios start from the same initial interval.
    all_x_ivls_initial = jax.tree_util.tree_map(
        lambda leaf: jnp.repeat(jnp.expand_dims(leaf, axis=0), num_scenarios, axis=0),
        x_interval
    )

    # 3. Define the function for the main scan over control stages
    def stage_scan_fn(carry_all_x_ivls, u_stage):
        # `carry_all_x_ivls`: PyTree of state intervals for all scenarios at the start of the stage.
        # `u_stage`: The single control vector to be applied for this stage.

        # 3a. Define an inner scan to simulate one stage (num_u_steps) for a single scenario
        def _propagate_one_scenario_for_stage(x_ivl_start, p_scenario):
            def _inner_scan_fn(x_ivl_carry, _):
                # The control `u_stage` is constant for this inner scan.
                x_ivl_next = propagate_interval_euler_admire(x_ivl_carry, u_stage, p_scenario, dt)
                return x_ivl_next, None
            
            # Run the simulation for num_u_steps and return the final state
            x_ivl_final, _ = jax.lax.scan(_inner_scan_fn, x_ivl_start, None, length=num_u_steps)
            return x_ivl_final

        # 3b. Use vmap to run the stage simulation for all scenarios in parallel
        all_x_ivls_end_of_stage = jax.vmap(_propagate_one_scenario_for_stage)(carry_all_x_ivls, all_params)

        # 3c. Calculate the pairwise loss at the end of this stage
        loss_at_stage = _calculate_pairwise_loss(all_x_ivls_end_of_stage, scaling_divisors)
        
        # 3d. The new carry is the set of final states, and the output is the stage loss
        return all_x_ivls_end_of_stage, loss_at_stage

    # 4. Run the main scan over the `N` control stages
    # The initial carry is the initial state for all scenarios.
    # The `xs` are the `N` control inputs from `u_ols`.
    _, all_stage_losses = jax.lax.scan(
        stage_scan_fn, all_x_ivls_initial, u_ols
    )
    
    # 5. The final loss is the product of the losses from each stage
    return jnp.prod(all_stage_losses)


In [5]:
jax_grad_staged = jax.grad(loss_ff_staged_product_jax, argnums=0)

def calculate_optimal_staged(x_interval, learning_rate, u_initial, p_nominal, p_faults, dt, num_u_steps=5, num_gd_steps=10):
    # # Define the body of the gradient descent loop
    p_faults_stacked = jax.tree_util.tree_map(lambda *leaves: jnp.stack(leaves), *p_faults)
    def gradient_step(i, u_current):
        grad = jax_grad_staged(u_current, x_interval, p_nominal, p_faults_stacked, dt, num_u_steps)
        return u_current - learning_rate * grad
    

    u_opt = jax.lax.fori_loop(0, num_gd_steps, gradient_step, u_initial)

    return u_opt


In [6]:
p_faults_list = [p_rc_ivl, p_lc_ivl, p_roe_ivl, p_rie_ivl, p_lie_ivl, p_loe_ivl, p_flap_ivl, p_rudder_ivl, p_yaw_tv_ivl, p_pitch_tv_ivl]


In [7]:
jit_find_optimal_staged = jit(partial(
    calculate_optimal_staged,
    num_u_steps=10,
    p_nominal=p_nominal_ivl,
    p_faults=p_faults_list,
    dt=0.05,
    num_gd_steps=num_steps
))


In [8]:
smart_u_2stage_initial_guess = jnp.array([[0., 0., 0., 0., 0., 0., 0., .1, 0., 0.],
                             [.1, .1, .1, .1, .1, .1, .1, .1, .1, .1]])


In [9]:
u_staged_opt = jax.block_until_ready(
    jit_find_optimal_staged(
        x0_interval_admire,
        1e-9,
        smart_u_2stage_initial_guess
    )
)


In [23]:
import time

In [32]:
t0 = time.time()
u_staged_opt = jit_find_optimal_staged(
        x0_interval_admire,
        1e-10,
        smart_u_2stage_initial_guess
    )
t1 = time.time()
print(f"Run time: {t1-t0}s")

Run time: 0.007799625396728516s


In [21]:
import timeit
timeit.timeit("u_staged_opt = jit_find_optimal_staged( \
        x0_interval_admire, \
        1e-10, \
        smart_u_2stage_initial_guess \
    )")

NameError: name 'jit_find_optimal_staged' is not defined

In [16]:
print(u_staged_opt)

[[-3.98568009e-05  1.74387125e-03  2.34414265e-02 -7.57531868e-03
  -1.07876845e-02  2.07514763e-02 -4.28388454e-03  1.15011916e-01
  -1.59838051e-02 -8.15454777e-03]
 [ 1.00122243e-01  1.00998186e-01  1.20139740e-01  9.35535952e-02
   8.86978656e-02  1.21432304e-01  9.79007334e-02  1.13496013e-01
   9.24252942e-02  9.44891423e-02]]


## 3D interval plots (ported from `admire_fault_demo.ipynb`)

In [33]:
# Extra imports needed for the 3D interval plots (not used by the optimization above).
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from typing import Any, Dict, List, Optional


In [34]:
# Human-readable names, in the same order as p_faults_list above.
fault_names = [
    "Right Canard Fault",
    "Left Canard Fault",
    "Right Outboard Elevon Fault",
    "Right Inboard Elevon Fault",
    "Left Inboard Elevon Fault",
    "Left Outboard Elevon Fault",
    "Flap Fault",
    "Rudder Fault",
    "Yaw Thrust Vectoring Fault",
    "Pitch Thrust Vectoring Fault",
]

dim_names = ["Velocity (m/s)", "AoA (rad)", "Sideslip Angle (rad)", "roll rate (rad/s)", "pitch rate (rad/s)",
             "yaw rate (rad/s)", "heading angle (rad)", "pitch angle (rad)", "roll angle (rad)"]


In [35]:
def simulate_staged_history(u_ols, x_interval, p_nominal, p_faults, fault_names, dt=0.1, num_u_steps=10):
    """
    Propagates the nominal and every fault scenario through the staged
    open-loop controls in u_ols, recording the state interval after every
    Euler step. Reuses propagate_interval_euler_admire defined above.

    Returns a dict keyed by scenario name: {"states": [Interval, ...]},
    the same "history" shape consumed by plot_3d_final_intervals and
    plot_3d_interval_history below.
    """
    assert len(p_faults) == len(fault_names), "Length of p_faults and fault_names must match."
    scenarios = ["Nominal"] + fault_names
    p_params = [p_nominal] + p_faults

    history = {name: {"states": [x_interval]} for name in scenarios}
    current_states = [x_interval for _ in scenarios]

    for u_stage in u_ols:
        for _ in range(num_u_steps):
            for j, name in enumerate(scenarios):
                current_states[j] = propagate_interval_euler_admire(current_states[j], u_stage, p_params[j], dt)
                history[name]["states"].append(current_states[j])

    return history


In [36]:
def plot_3d_final_intervals(
    history: Dict[str, Any],
    dim_names: List[str],
    color_map: Optional[Dict[str, str]] = None,
    save_to_pdf: bool = False,
    pdf_pages: Optional[PdfPages] = None
) -> None:
    """
    Generates a readable 3D plot of final state intervals for dimensions 4, 5, and 6,
    with larger fonts and auto-scaled tight axis limits.

    Args:
        history: The output dictionary from the simulation function.
        dim_names: A list of names for each state dimension for axis labeling.
        color_map: (Optional) A dictionary mapping scenario names to colors.
        save_to_pdf: If True, saves the plot to a PDF file.
        pdf_pages: (Optional) An existing PdfPages object to add the plot to.
    """
    # --- 1. Setup and Color Map Generation ---
    if color_map is None:
        scenarios = list(history.keys())
        fault_names = [name for name in scenarios if name != "Nominal"]
        cmap = plt.cm.get_cmap('viridis', len(fault_names))
        generated_colors = [cmap(i) for i in range(len(fault_names))]
        color_map = {name: color for name, color in zip(fault_names, generated_colors)}
        if "Nominal" in scenarios: color_map["Nominal"] = "blue"

    print("Generating 3D plot for dimensions 4, 5, and 6 (indices 3, 4, 5)...")
    scenarios = list(history.keys())
    final_intervals = {name: history[name]["states"][-1] for name in scenarios}

    # --- 2. Calculate tight axis limits dynamically ---
    min_x, max_x = float('inf'), float('-inf')
    min_y, max_y = float('inf'), float('-inf')
    min_z, max_z = float('inf'), float('-inf')

    for ivl in final_intervals.values():
        min_x = min(min_x, ivl.lower[3])
        max_x = max(max_x, ivl.upper[3])
        min_y = min(min_y, ivl.lower[4])
        max_y = max(max_y, ivl.upper[4])
        min_z = min(min_z, ivl.lower[5])
        max_z = max(max_z, ivl.upper[5])

    def get_padded_limits(min_val, max_val, padding_factor=0.05):
        range_val = max_val - min_val
        if range_val == 0: range_val = abs(max_val) * 0.1 or 0.1
        padding = range_val * padding_factor
        return [min_val - padding, max_val + padding]

    x_lims = get_padded_limits(min_x, max_x)
    y_lims = get_padded_limits(min_y, max_y)
    z_lims = get_padded_limits(min_z, max_z)

    # --- 3. Plotting Setup and Loop ---
    fig_3d = plt.figure(figsize=(12, 10))
    ax_3d = fig_3d.add_subplot(111, projection='3d')

    def create_cuboid_faces(x_range, y_range, z_range):
        x0,x1=x_range; y0,y1=y_range; z0,z1=z_range
        v=[(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),(x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
        f=[[v[0],v[1],v[2],v[3]],[v[4],v[5],v[6],v[7]],[v[0],v[1],v[5],v[4]],
           [v[2],v[3],v[7],v[6]],[v[0],v[3],v[7],v[4]],[v[1],v[2],v[6],v[5]]]
        return f

    for name in scenarios:
        final_ivl = final_intervals[name]
        collection = Poly3DCollection(
            create_cuboid_faces(
                (final_ivl.lower[3], final_ivl.upper[3]),
                (final_ivl.lower[4], final_ivl.upper[4]),
                (final_ivl.lower[5], final_ivl.upper[5])
            ),
            facecolors=color_map[name], linewidths=1, edgecolors='k', alpha=0.20
        )
        ax_3d.add_collection3d(collection)

    # --- 4. Apply limits and larger fonts ---
    TITLE_FONTSIZE = 32
    LABEL_FONTSIZE = 20
    TICK_FONTSIZE = 16

    ax_3d.set_xlim(min_x, max_x)
    ax_3d.set_ylim(min_y, max_y)
    ax_3d.set_zlim(min_z, max_z)

    ax_3d.set_xlabel(dim_names[3], fontsize=LABEL_FONTSIZE, labelpad=15)
    ax_3d.set_ylabel(dim_names[4], fontsize=LABEL_FONTSIZE, labelpad=15)
    ax_3d.set_zlabel(dim_names[5], fontsize=LABEL_FONTSIZE, labelpad=15)

    ax_3d.tick_params(axis='x', labelsize=TICK_FONTSIZE)
    ax_3d.tick_params(axis='y', labelsize=TICK_FONTSIZE)
    ax_3d.tick_params(axis='z', labelsize=TICK_FONTSIZE)

    legend_patches = [Rectangle((0, 0), 1, 1, fc=color_map[name], alpha=0.5, label=name) for name in scenarios]
    ax_3d.legend(handles=legend_patches, fontsize=TICK_FONTSIZE, loc='upper right', bbox_to_anchor=(0.95, 1.0))
    ax_3d.grid(True)
    plt.tight_layout()

    # --- 5. Save/Display ---
    if save_to_pdf:
        create_own_pdf = pdf_pages is None
        if create_own_pdf:
            default_filename = "figure_3_reproduction.pdf"
            print(f"No PdfPages object provided. Saving to new file: {default_filename}")
            pdf_pages = PdfPages(default_filename)
        pdf_pages.savefig(fig_3d, bbox_inches='tight', pad_inches=0)
        if create_own_pdf:
            pdf_pages.close()
            print("PDF generation complete.")
    else:
        plt.show()
    plt.close(fig_3d)


In [37]:
def plot_3d_interval_history(
    history: Dict[str, Any],
    dim_names: List[str],
    initial_interval: Optional["irx.Interval"] = None,
    color_map: Optional[Dict[str, str]] = None,
    save_to_pdf: bool = False,
    pdf_pages: Optional[PdfPages] = None
) -> None:
    """
    Generates a 3D plot of state interval history, starting from a given initial interval.
    The history is rendered as a smooth, faint tube, and the final state
    is shown as a more solid, concrete box.

    Args:
        history: The output dictionary from the simulation function.
        dim_names: A list of names for each state dimension for axis labeling.
        initial_interval: (Optional) A specific interval to use as the starting
                          point for all history tubes.
        color_map: (Optional) A dictionary mapping scenario names to colors.
        save_to_pdf: If True, saves the plot to a PDF file.
        pdf_pages: (Optional) An existing PdfPages object to add the plot to.
    """
    # --- 1. Setup and Color Map Generation ---
    if color_map is None:
        scenarios = list(history.keys())
        fault_names = [name for name in scenarios if name != "Nominal"]
        cmap = plt.cm.get_cmap('viridis', len(fault_names))
        generated_colors = [cmap(i) for i in range(len(fault_names))]
        color_map = {name: color for name, color in zip(fault_names, generated_colors)}
        if "Nominal" in scenarios: color_map["Nominal"] = "blue"

    print("Generating smooth 3D history plot for dimensions 4, 5, and 6...")
    scenarios = list(history.keys())

    # --- 2. Axis limit calculation (includes initial_interval) ---
    min_x, max_x = float('inf'), float('-inf')
    min_y, max_y = float('inf'), float('-inf')
    min_z, max_z = float('inf'), float('-inf')

    if initial_interval:
        min_x = min(min_x, initial_interval.lower[3]); max_x = max(max_x, initial_interval.upper[3])
        min_y = min(min_y, initial_interval.lower[4]); max_y = max(max_y, initial_interval.upper[4])
        min_z = min(min_z, initial_interval.lower[5]); max_z = max(max_z, initial_interval.upper[5])

    for name in scenarios:
        for ivl in history[name]["states"]:
            min_x = min(min_x, ivl.lower[3]); max_x = max(max_x, ivl.upper[3])
            min_y = min(min_y, ivl.lower[4]); max_y = max(max_y, ivl.upper[4])
            min_z = min(min_z, ivl.lower[5]); max_z = max(max_z, ivl.upper[5])

    def get_padded_limits(min_val, max_val, padding_factor=0.05):
        range_val = max_val - min_val
        if range_val == 0: range_val = abs(max_val) * 0.1 or 0.1
        padding = range_val * padding_factor
        return [min_val - padding, max_val + padding]

    x_lims = get_padded_limits(min_x, max_x)
    y_lims = get_padded_limits(min_y, max_y)
    z_lims = get_padded_limits(min_z, max_z)

    # --- 3. Plotting setup and helpers ---
    fig_3d = plt.figure(figsize=(12, 10))
    ax_3d = fig_3d.add_subplot(111, projection='3d')

    def get_cuboid_vertices(ivl):
        x0, x1 = ivl.lower[3], ivl.upper[3]
        y0, y1 = ivl.lower[4], ivl.upper[4]
        z0, z1 = ivl.lower[5], ivl.upper[5]
        return [(x0,y0,z0), (x1,y0,z0), (x1,y1,z0), (x0,y1,z0),
                (x0,y0,z1), (x1,y0,z1), (x1,y1,z1), (x0,y1,z1)]

    def create_cuboid_faces(vertices):
        return [[vertices[0], vertices[1], vertices[2], vertices[3]], [vertices[4], vertices[5], vertices[6], vertices[7]],
                [vertices[0], vertices[1], vertices[5], vertices[4]], [vertices[2], vertices[3], vertices[7], vertices[6]],
                [vertices[0], vertices[3], vertices[7], vertices[4]], [vertices[1], vertices[2], vertices[6], vertices[5]]]

    def create_tube_segment_faces(v1, v2):
        return [[v1[0],v1[1],v2[1],v2[0]], [v1[4],v1[5],v2[5],v2[4]], [v1[0],v1[4],v2[4],v2[0]],
                [v1[3],v1[7],v2[7],v2[3]], [v1[1],v1[5],v2[5],v2[1]], [v1[2],v1[6],v2[6],v2[2]]]

    # --- 4. Plot tubes, starting from the initial interval when provided ---
    for name in scenarios:
        all_states = history[name]["states"]

        tube_path = ([initial_interval] + all_states) if initial_interval else all_states
        if len(tube_path) < 2:
            if all_states:
                final_ivl = all_states[-1]
                final_verts = get_cuboid_vertices(final_ivl)
                final_box = Poly3DCollection(create_cuboid_faces(final_verts), facecolors=color_map[name],
                                             linewidths=1, edgecolors='k', alpha=0.5)
                ax_3d.add_collection3d(final_box)
            continue

        # --- Plot the history as a faint, smooth tube ---
        tube_faces = []
        start_cap_verts = get_cuboid_vertices(tube_path[0])
        tube_faces.extend(create_cuboid_faces(start_cap_verts))

        for i in range(len(tube_path) - 1):
            v1 = get_cuboid_vertices(tube_path[i])
            v2 = get_cuboid_vertices(tube_path[i+1])
            tube_faces.extend(create_tube_segment_faces(v1, v2))

        tube_collection = Poly3DCollection(
            tube_faces, facecolors=color_map[name], linewidths=0, alpha=0.15
        )
        ax_3d.add_collection3d(tube_collection)

        # --- Plot the actual final interval as a more solid box ---
        if all_states:
            final_ivl = all_states[-1]
            final_verts = get_cuboid_vertices(final_ivl)
            final_box_collection = Poly3DCollection(
                create_cuboid_faces(final_verts), facecolors=color_map[name],
                linewidths=1, edgecolors='k', alpha=0.5
            )
            ax_3d.add_collection3d(final_box_collection)

    # --- 5. Formatting, limits, and labels ---
    TITLE_FONTSIZE = 32; LABEL_FONTSIZE = 16; TICK_FONTSIZE = 16
    ax_3d.set_xlim(x_lims); ax_3d.set_ylim(y_lims); ax_3d.set_zlim(z_lims)
    ax_3d.set_xlabel(dim_names[3], fontsize=LABEL_FONTSIZE, labelpad=15)
    ax_3d.set_ylabel(dim_names[4], fontsize=LABEL_FONTSIZE, labelpad=15)
    ax_3d.set_zlabel(dim_names[5], fontsize=LABEL_FONTSIZE, labelpad=50)
    ax_3d.tick_params(axis='x', labelsize=TICK_FONTSIZE)
    ax_3d.tick_params(axis='y', labelsize=TICK_FONTSIZE)
    ax_3d.tick_params(axis='z', labelsize=TICK_FONTSIZE)
    legend_patches = [Rectangle((0, 0), 1, 1, fc=color_map[name], alpha=0.6, label=name) for name in scenarios]
    ax_3d.legend(handles=legend_patches, fontsize=TICK_FONTSIZE, loc='upper left', bbox_to_anchor=(-0.1, 0.95))
    ax_3d.grid(True)
    plt.tight_layout()

    # --- 6. Save/Display ---
    if save_to_pdf:
        create_own_pdf = pdf_pages is None
        if create_own_pdf:
            default_filename = "figure_3_reproduction.pdf"
            print(f"No PdfPages object provided. Saving to new file: {default_filename}")
            pdf_pages = PdfPages(default_filename)
        pdf_pages.savefig(fig_3d, bbox_inches='tight', pad_inches=0.)
        if create_own_pdf:
            pdf_pages.close()
            print("PDF generation complete.")
    else:
        plt.show()
    plt.close(fig_3d)


In [38]:
# Propagate every scenario under the optimized staged controls, then render
# both 3D plots into a single PDF named "figure_3_reproduction.pdf".
history_3d = simulate_staged_history(
    u_ols=u_staged_opt,
    x_interval=x0_interval_admire,
    p_nominal=p_nominal_ivl,
    p_faults=p_faults_list,
    fault_names=fault_names,
    dt=0.025,
    num_u_steps=20,
)

pdf_filename = "figure_3_reproduction.pdf"
pdf_pages = PdfPages(pdf_filename)
pdf_pages.infodict()["Title"] = "figure_3_reproduction.pdf"

plot_3d_final_intervals(
    history=history_3d,
    dim_names=dim_names,
    save_to_pdf=True,
    color_map=None,
    pdf_pages=pdf_pages,
)

plot_3d_interval_history(
    history=history_3d,
    dim_names=dim_names,
    initial_interval=x0_interval_admire,
    save_to_pdf=True,
    color_map=None,
    pdf_pages=pdf_pages,
)

pdf_pages.close()
print(f"Saved {pdf_filename}")


/tmp/ipykernel_8870/4122441249.py:23: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('viridis', len(fault_names))


Generating 3D plot for dimensions 4, 5, and 6 (indices 3, 4, 5)...


/tmp/ipykernel_8870/83474079.py:27: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('viridis', len(fault_names))


Generating smooth 3D history plot for dimensions 4, 5, and 6...
Saved figure_3_reproduction.pdf
